# Make Models

B Steele, Colorado State University, ROSSyndicate — last updated Sep 15, 2026

This notebook trains the production Secchi-depth model: an ensemble of XGBoost and LightGBM models over the five independent HUC8-random CV arrangements built in `03_split_data.ipynb`, evaluated on that notebook's one fixed, never-trained-on holdout.

This replaces the original `04_make_models.Rmd`'s single-split, XGBoost-only, leave-one-partition-out pipeline. What changed, and why (see project memory `workflow-v3-findings` for the full research record):

- **Five independent CV arrangements, not one.** Each ensemble seed (`501`–`505`) runs its own complete feature-selection + tuning + training pipeline, entirely independently, on its own 4-fold split of the shared CV pool. The final production prediction is the average across all 5 seeds × 2 models × 4 folds — 40 models.
- **Gap-aware tuning at every hyperparameter-selection stage**, not just the final one. Random-search hyperparameter selection picks the smallest train/validation RMSE gap among near-optimal trials (within 2% of the best validation RMSE), rather than always the single lowest validation RMSE — free consistency win in the research pipeline, no accuracy cost.
- **Unweighted training by default.** The original SDD-weighted (toward high-SDD values) scheme was re-tested on this workflow's own final configuration and found to net a ~0.6% overall regression (it helps the sparse top-quartile tail but hurts the well-populated rest more). `python/weighting.py` keeps it available as a documented tail-priority alternative, but it is not used here.
- **XGBoost + LightGBM.** The original pipeline used only XGBoost. LightGBM is now included as a second model type, and the final production prediction is the average of both. This is a deliberate ensemble design choice, not a hyperparameter search; the two models are trained with the same features and hyperparameters, and their predictions are averaged at the end.
- **The feature-group ablation (optical-only vs. +site vs. +weather vs. +both) is not re-run here.** That question is already answered — site and weather features measurably help — so this notebook goes straight to the combined candidate pool. This is the one deliberate simplification relative to the full research pipeline, made to keep a from-scratch production run tractable; every other stage (correlation pruning, backward elimination, gap-aware tuning, final training) runs in full.

## Setup

In [ ]:
import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path("python").resolve()))
import backward_elim as be
import features
import metrics as M
import model_lgb
import model_xgb
import spatial_cv
from tuning import gap_aware_tune

be.register_model("xgboost", model_xgb)
be.register_model("lightgbm", model_lgb)

AQUAMATCH_DIR = Path("aquamatch_files")
MODELS_DIR = Path("xg_models") / "v3_production"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "harmonized_value"
MODELS = {"xgboost": model_xgb, "lightgbm": model_lgb}
N_TRIALS_INITIAL = {"xgboost": 25, "lightgbm": 25}
N_TRIALS_FINAL = {"xgboost": 40, "lightgbm": 40}
ELIM_TOL = 0.005
ELIM_SEED = 47
TUNE_SEED = 47

SITE_COLS = ["elevation_m", "catchment_area_sqkm", "pct_impervious_2006", "pct_urban_2006",
             "pct_forest_2006", "pct_cropland_2006", "pct_wetland_2006", "shore_flag"]
WEATHER_COLS_CURATED = ["precip_mm_prev30", "tmax_degC_prev30"]


def log(seed, msg):
    print(f"[{time.strftime('%H:%M:%S')}][seed{seed}] {msg}", flush=True)

## Engineer spectral indices

Build the curated band-ratio/index candidate features (`python/features.py`) from the sensor-corrected bands assembled in `03_split_data.ipynb`. Also applies `coarsen_site_features()`, which log-bins `catchment_area_sqkm` and rounds the `pct_*_2006` land-cover fractions to the nearest percentage point, so a heavily-resampled single waterbody's static, per-comid LakeCat values can't act as a de facto waterbody ID the model keys a memorized SDD value off of. This does not address the separate, already-documented leakage from waterbodies spanning more than one HUC8 (Lake Powell chief among them) — those remain large and distinct enough to stay identifiable after coarsening by design.

In [ ]:
df = pd.read_parquet(AQUAMATCH_DIR / "base_with_splits.parquet")
df = features.add_spectral_indices(df)
df = features.coarsen_site_features(df)
print(f"{len(df):,} rows, {df.shape[1]} columns after spectral-index engineering")

## Per-seed pipeline

For each ensemble seed, independently:

1. **Candidate pool.** Optical/spectral candidates (bands + indices + `atm_corr_LaSRC`) plus `SITE_COLS` (including `shore_flag`) plus the curated antecedent-weather columns, correlation-pruned (`|r| > 0.95` clusters collapsed to the member most correlated with the target).
2. **Initial gap-aware tuning** on the full pruned candidate pool, to get a fixed, reasonable hyperparameter set for the elimination stage below (elimination itself holds hyperparameters fixed while comparing feature subsets, for tractability).
3. **Backward elimination** — per-model (shadow-decoy stopping against a permutation-importance noise floor) and jointly across both models — down to each model's own final feature set; take the **intersection** as this seed's production feature set.
4. **Final gap-aware tuning** on the intersection feature set (more trials than step 2, since this is the configuration actually deployed).
5. **Final training**: 4 fold models per algorithm, unweighted, saved to disk; CV metrics (out-of-fold, across all 4 folds) and holdout metrics (the one fixed holdout, never touched until this point) recorded; per-row holdout predictions saved for ensembling in `05_evaluate_ensemble.ipynb`.

In [ ]:
def run_seed_pipeline(seed, df):
    out_dir = MODELS_DIR / f"seed{seed}"
    out_dir.mkdir(parents=True, exist_ok=True)
    t_start = time.time()

    folds = spatial_cv.build_folds(df, seed)
    test = df[df["is_holdout"]].reset_index(drop=True)
    log(seed, f"folds: {[(f.part, len(f.train), len(f.val)) for f in folds]}, holdout n={len(test)}")

    # --- candidate pool + correlation pruning (this seed's own train_val pool) ---
    metadata_cols = {"HUC8", "holdout_part", "is_holdout"} | {f"cvfold_seed{s}" for s in spatial_cv.ENSEMBLE_SEEDS}
    extra_cols = set(SITE_COLS) | set(WEATHER_COLS_CURATED) | metadata_cols
    optical_feats = [f for f in features.candidate_feature_list(df) if f not in extra_cols]
    candidates = optical_feats + SITE_COLS + WEATHER_COLS_CURATED

    train_val = pd.concat([f.train for f in folds] + [folds[0].val], ignore_index=True).drop_duplicates(
        subset=["siteSR_id", "date"])
    pruned = features.correlation_prune(train_val, candidates, target=TARGET)
    log(seed, f"candidate pool: {len(candidates)} -> {len(pruned)} after correlation pruning")
    log(seed, f"pruned set: {pruned}")

    # --- stage 1: initial gap-aware tuning (fixed params for elimination) ---
    fixed_params = {}
    for model_name, mod in MODELS.items():
        log(seed, f"=== initial gap-aware tuning: {model_name} on {len(pruned)}-feature pool ===")
        result = gap_aware_tune(mod, folds, pruned, TARGET, n_trials=N_TRIALS_INITIAL[model_name], seed=47)
        fixed_params[model_name] = result["best_params"]
        log(seed, f"{model_name}: val_rmse={result['best_score']:.4f} gap={result['best_gap']:.4f}")

    # --- stage 2: backward elimination (per-model + joint) ---
    per_model = {}
    for model_name in MODELS:
        log(seed, f"=== backward elimination: {model_name} ===")
        result = be.backward_eliminate(model_name, folds, TARGET, fixed_params[model_name], pruned,
                                        weight_fn=None, tol=ELIM_TOL, seed=ELIM_SEED,
                                        log=lambda m: log(seed, m))
        per_model[model_name] = result
        log(seed, f"{model_name} final ({len(result['final_features'])}): {result['final_features']}")

    log(seed, "=== joint elimination (xgboost + lightgbm) ===")
    folds_by_model = {m: folds for m in MODELS}
    joint_result = be.joint_backward_eliminate(list(MODELS), folds_by_model, TARGET, fixed_params, pruned,
                                                weight_fn=None, tol=ELIM_TOL, seed=ELIM_SEED,
                                                log=lambda m: log(seed, m))
    log(seed, f"joint final ({len(joint_result['final_features'])}): {joint_result['final_features']}")

    intersection = sorted(set(per_model["xgboost"]["final_features"]) & set(per_model["lightgbm"]["final_features"]))
    union = sorted(set(per_model["xgboost"]["final_features"]) | set(per_model["lightgbm"]["final_features"]))
    log(seed, f"intersection ({len(intersection)}): {intersection}")

    with open(out_dir / "backward_elim_summary.json", "w") as fh:
        json.dump(dict(
            starting_candidates=candidates, correlation_pruned=pruned,
            per_model_final={m: per_model[m]["final_features"] for m in MODELS},
            per_model_path={m: per_model[m]["path"] for m in MODELS},
            joint_final=joint_result["final_features"], joint_path=joint_result["path"],
            intersection_of_per_model=intersection, union_of_per_model=union,
        ), fh, indent=2, default=str)

    # --- stage 3: final gap-aware tuning on the intersection feature set ---
    tuned = {}
    for model_name, mod in MODELS.items():
        log(seed, f"=== final gap-aware tuning: {model_name} on {len(intersection)}-feature intersection ===")
        result = gap_aware_tune(mod, folds, intersection, TARGET, n_trials=N_TRIALS_FINAL[model_name], seed=TUNE_SEED)
        log(seed, f"{model_name} chosen: val_rmse={result['best_score']:.4f} gap={result['best_gap']:.4f}")
        tuned[model_name] = result
        with open(out_dir / f"final_tune_{model_name}.json", "w") as fh:
            json.dump({"model": model_name, "features": intersection, **result}, fh, indent=2, default=str)

    # --- stage 4: final unweighted training + evaluation ---
    summary = {}
    holdout_rows = []
    for model_name, mod in MODELS.items():
        params = tuned[model_name]["best_params"]
        fold_models = mod.train_fold_models(folds, intersection, TARGET, params)

        for i, fm in enumerate(fold_models):
            ext = "json" if model_name == "xgboost" else "txt"
            fm.save_model(str(out_dir / f"{model_name}_fold{i+1}.{ext}"))

        cv_preds, cv_ys = [], []
        for f, model in zip(folds, fold_models):
            cv_preds.append(mod.predict_ensemble([model], f.val[intersection]))
            cv_ys.append(f.val[TARGET].values)
        cv_metrics = M.all_metrics(np.concatenate(cv_ys), np.concatenate(cv_preds))

        test_pred = mod.predict_ensemble(fold_models, test[intersection])
        test_metrics = M.all_metrics(test[TARGET].values, test_pred)
        log(seed, f"{model_name} FINAL: cv_rmse={cv_metrics['rmse']:.4f} holdout_rmse={test_metrics['rmse']:.4f}")

        summary[model_name] = dict(features=intersection, params=params, cv_metrics=cv_metrics,
                                    test_metrics=test_metrics)

        r = test[["siteSR_id", "date", "HUC4", "HUC8"]].copy()
        r["seed"] = seed
        r["model"] = model_name
        r["y"] = test[TARGET].values
        r["pred"] = test_pred
        holdout_rows.append(r)

    with open(out_dir / "final_eval_summary.json", "w") as fh:
        json.dump(summary, fh, indent=2, default=str)
    holdout_df = pd.concat(holdout_rows, ignore_index=True)
    holdout_df.to_parquet(out_dir / "holdout_predictions.parquet")

    log(seed, f"SEED {seed} DONE in {time.time()-t_start:.0f}s")
    return summary

In [ ]:
all_summaries = {}
for seed in spatial_cv.ENSEMBLE_SEEDS:
    all_summaries[seed] = run_seed_pipeline(seed, df)

## Per-seed summary

Each seed selects its own feature set and hyperparameters independently; this table is a sanity check on cross-seed spread before ensembling in `05_evaluate_ensemble.ipynb`, not the production number itself (the production number is the 40-model ensemble average, evaluated on the shared holdout, computed in that next notebook).

In [ ]:
rows = []
for seed, summary in all_summaries.items():
    for model_name, s in summary.items():
        rows.append(dict(seed=seed, model=model_name, n_features=len(s["features"]),
                          cv_rmse=s["cv_metrics"]["rmse"], holdout_rmse=s["test_metrics"]["rmse"],
                          holdout_bias=s["test_metrics"]["bias"]))
pd.DataFrame(rows).sort_values(["model", "seed"])